In [1]:
import os
from ast import literal_eval
import pandas as pd
import spacy
import re
from spacy.tokenizer import Tokenizer
import matplotlib.pyplot as plt
import dill
import numpy as np

In [2]:
data = pd.read_csv('../../data/PFUB_ERLASS_dataset_processed_with_geschaeftsnummer.csv')

In [3]:
data = data[data['is_pfub']==True].reset_index(drop=True)

In [4]:
data

,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,tokenized,text_w_tags
0,Amtsgericht Memmingen\nAbteilung für Zwangsvol...,01.04.2025/ocr-v2_54_M_1067_25_Schreiben_vom_3...,approved_attachment_and_transfer_order,amtsgericht memmingen\nabteilung für zwangsvol...,"{'case_slug': '163464253141', 'debtor_name': '...",True,False,"['amtsgericht', 'memmingen', 'abteilung', 'zwa...",amtsgericht memmingen\nabteilung für zwangsvol...
1,Amtsgericht Frankfurt am Main\nMobiliarzwangsv...,01.06.2022/1654077659_Doc_01062022_000000001_1...,approved_seizure,amtsgericht frankfurt am main\nmobiliarzwangsv...,"{'case_slug': '110872558124', 'debtor_name': '...",True,False,"['amtsgericht', 'frankfurt', 'main', 'mobiliar...",amtsgericht frankfurt am main\nmobiliarzwangsv...
2,Amtsgericht Eschweiler\n070071 967072\n-Geschä...,01.06.2022/1654077662_Doc_01062022_000000001_1...,approved_seizure,amtsgericht eschweiler\n070071 967072\n-geschä...,"{'case_slug': '114249164962', 'debtor_name': '...",True,False,"['amtsgericht', 'eschweiler', '-geschäftsstell...",amtsgericht eschweiler\n<NUMBER><PHONE>\n-gesc...
3,Amtsgericht Münster\n-33- Amtsgericht Münster ...,01.07.2024/ocr-v2_BB_005_01072024_122825.pdf,approved_seizure,amtsgericht münster\n-33- amtsgericht münster ...,"{'case_slug': '136916228754', 'debtor_name': '...",True,False,"['amtsgericht', 'münster', 'amtsgericht', 'mün...",amtsgericht münster\n<NUMBER>- amtsgericht mün...
4,Amtsgericht Viersen\n-Geschäftsstelle-\n-15- A...,01.07.2025/ocr-v2_Allgemeines_Schreiben_R__202...,approved_attachment_and_transfer_order,amtsgericht viersen\n-geschäftsstelle-\n-15- a...,"{'case_slug': '125495503778', 'debtor_name': '...",True,False,"['amtsgericht', 'viersen', '-geschäftsstelle-'...",amtsgericht viersen\n-geschäftsstelle-\n<NUMBE...
...,...,...,...,...,...,...,...,...,...
461,Amtsgericht Wuppertal\n-44- Amtsgericht Wupper...,31.01.2023/1675166710_YA_003_31012023_124814.pdf,approved_seizure,amtsgericht wuppertal\n-44- amtsgericht wupper...,"{'case_slug': '110454161073', 'debtor_name': '...",True,False,"['amtsgericht', 'wuppertal', 'amtsgericht', 'w...",amtsgericht wuppertal\n<NUMBER>- amtsgericht w...
462,"Amtsgericht Itzehoe\nAmtsgericht Itzehoe, Berg...",31.01.2023/1675166710_YA_003_31012023_124815.pdf,approved_seizure,"amtsgericht itzehoe\namtsgericht itzehoe, berg...","{'case_slug': None, 'debtor_name': None, 'cred...",True,False,"['amtsgericht', 'itzehoe', 'amtsgericht', 'itz...","amtsgericht itzehoe\namtsgericht itzehoe, berg..."
463,Amtsgericht Leverkusen\n-45- Amtsgericht Lever...,31.05.2023/1685534803_PF_005_31052023_140508.pdf,approved_seizure,amtsgericht leverkusen\n-45- amtsgericht lever...,"{'case_slug': '117578968598', 'debtor_name': '...",True,False,"['amtsgericht', 'leverkusen', 'amtsgerichen', ...",amtsgericht leverkusen\n<NUMBER>- amtsgericht ...
464,Amtsgericht Frankfurt am Main\nMobiliarzwangsv...,31.07.2023/1690794316_KD_004_31072023_110320.pdf,approved_seizure,amtsgericht frankfurt am main\nmobiliarzwangsv...,"{'case_slug': '125677855012', 'debtor_name': '...",True,False,"['amtsgericht', 'frankfurt', 'main', 'mobiliar...",amtsgericht frankfurt am main\nmobiliarzwangsv...


In [5]:
def extract_data(data):
    data = literal_eval(data)
    case_slug = data['case_slug'] if 'case_slug' in data else None
    debtor_name = data['debtor_name'] if 'debtor_name' in data else None

    return case_slug, debtor_name

data[['case_slug','debtor_name']] = data['data'].apply(lambda x: pd.Series(extract_data(x)))

In [6]:
data

,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,tokenized,text_w_tags,case_slug,debtor_name
0,Amtsgericht Memmingen\nAbteilung für Zwangsvol...,01.04.2025/ocr-v2_54_M_1067_25_Schreiben_vom_3...,approved_attachment_and_transfer_order,amtsgericht memmingen\nabteilung für zwangsvol...,"{'case_slug': '163464253141', 'debtor_name': '...",True,False,"['amtsgericht', 'memmingen', 'abteilung', 'zwa...",amtsgericht memmingen\nabteilung für zwangsvol...,163464253141,Julia Zauner
1,Amtsgericht Frankfurt am Main\nMobiliarzwangsv...,01.06.2022/1654077659_Doc_01062022_000000001_1...,approved_seizure,amtsgericht frankfurt am main\nmobiliarzwangsv...,"{'case_slug': '110872558124', 'debtor_name': '...",True,False,"['amtsgericht', 'frankfurt', 'main', 'mobiliar...",amtsgericht frankfurt am main\nmobiliarzwangsv...,110872558124,"Kaiser, Vanessa"
2,Amtsgericht Eschweiler\n070071 967072\n-Geschä...,01.06.2022/1654077662_Doc_01062022_000000001_1...,approved_seizure,amtsgericht eschweiler\n070071 967072\n-geschä...,"{'case_slug': '114249164962', 'debtor_name': '...",True,False,"['amtsgericht', 'eschweiler', '-geschäftsstell...",amtsgericht eschweiler\n<NUMBER><PHONE>\n-gesc...,114249164962,Pitschel
3,Amtsgericht Münster\n-33- Amtsgericht Münster ...,01.07.2024/ocr-v2_BB_005_01072024_122825.pdf,approved_seizure,amtsgericht münster\n-33- amtsgericht münster ...,"{'case_slug': '136916228754', 'debtor_name': '...",True,False,"['amtsgericht', 'münster', 'amtsgericht', 'mün...",amtsgericht münster\n<NUMBER>- amtsgericht mün...,136916228754,Georgina Kaiser
4,Amtsgericht Viersen\n-Geschäftsstelle-\n-15- A...,01.07.2025/ocr-v2_Allgemeines_Schreiben_R__202...,approved_attachment_and_transfer_order,amtsgericht viersen\n-geschäftsstelle-\n-15- a...,"{'case_slug': '125495503778', 'debtor_name': '...",True,False,"['amtsgericht', 'viersen', '-geschäftsstelle-'...",amtsgericht viersen\n-geschäftsstelle-\n<NUMBE...,125495503778,Sultan Aslan
...,...,...,...,...,...,...,...,...,...,...,...
461,Amtsgericht Wuppertal\n-44- Amtsgericht Wupper...,31.01.2023/1675166710_YA_003_31012023_124814.pdf,approved_seizure,amtsgericht wuppertal\n-44- amtsgericht wupper...,"{'case_slug': '110454161073', 'debtor_name': '...",True,False,"['amtsgericht', 'wuppertal', 'amtsgericht', 'w...",amtsgericht wuppertal\n<NUMBER>- amtsgericht w...,110454161073,Magner
462,"Amtsgericht Itzehoe\nAmtsgericht Itzehoe, Berg...",31.01.2023/1675166710_YA_003_31012023_124815.pdf,approved_seizure,"amtsgericht itzehoe\namtsgericht itzehoe, berg...","{'case_slug': None, 'debtor_name': None, 'cred...",True,False,"['amtsgericht', 'itzehoe', 'amtsgericht', 'itz...","amtsgericht itzehoe\namtsgericht itzehoe, berg...",None,None
463,Amtsgericht Leverkusen\n-45- Amtsgericht Lever...,31.05.2023/1685534803_PF_005_31052023_140508.pdf,approved_seizure,amtsgericht leverkusen\n-45- amtsgericht lever...,"{'case_slug': '117578968598', 'debtor_name': '...",True,False,"['amtsgericht', 'leverkusen', 'amtsgerichen', ...",amtsgericht leverkusen\n<NUMBER>- amtsgericht ...,117578968598,Aschenbach
464,Amtsgericht Frankfurt am Main\nMobiliarzwangsv...,31.07.2023/1690794316_KD_004_31072023_110320.pdf,approved_seizure,amtsgericht frankfurt am main\nmobiliarzwangsv...,"{'case_slug': '125677855012', 'debtor_name': '...",True,False,"['amtsgericht', 'frankfurt', 'main', 'mobiliar...",amtsgericht frankfurt am main\nmobiliarzwangsv...,125677855012,"Tiropoulos, Konstantinos"


In [7]:
data.dropna(subset="case_slug", inplace=True)

In [8]:
slug_lengths = data['case_slug'].apply(lambda x: len(x))

In [9]:
print("Distinct slug lengths:", slug_lengths.unique())

Distinct slug lengths: [12 21 20 11  4]


In [10]:
# check the lentgs 21,20 and 4

data_21 = data[slug_lengths==21]
data_20 = data[slug_lengths==20]
data_4 = data[slug_lengths==4]

In [11]:
def print_out_clickable_link(df):
    for index, row in df.iterrows():
        print(f"Index: {index}, Case Slug: {row['case_slug']}, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix={row['object_key']}")
print_out_clickable_link(data_21)
# THE DATA WITH LENGTH 21 IS OLD SLUGS (2023,2022. LOOKS LIKE THIS: 257 47 l22DH01iSK1 0991 531 4966)

# print_out_clickable_link(data_20)
# SAME THING FOR LENGTH 20. E.G: 3962/22SK01/VK14665532494

# print_out_clickable_link(data_4)
# this looks okay, the ocr extraction is wrong but the document slug is okay

Index: 128, Case Slug: 334112201121814159442, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=09.06.2023/1686315969_KD_002_09062023_145003.pdf
Index: 202, Case Slug: 219442101111810161419, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=15.11.2022/1668506758_Scan_YA_00215112022_104157.pdf
Index: 297, Case Slug: 244272201107454953657, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=21.11.2022/1669023316_SK_002_21112022_100624.pdf
Index: 356, Case Slug: 257472201109915314966, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=25.07.2023/1690275941_KD_005_25072023_103950.pdf


In [12]:
drop_idx = set(data_21.index) | set(data_20.index)
data = data.drop(index=drop_idx).reset_index(drop=True)

In [13]:
data

,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,tokenized,text_w_tags,case_slug,debtor_name
0,Amtsgericht Memmingen\nAbteilung für Zwangsvol...,01.04.2025/ocr-v2_54_M_1067_25_Schreiben_vom_3...,approved_attachment_and_transfer_order,amtsgericht memmingen\nabteilung für zwangsvol...,"{'case_slug': '163464253141', 'debtor_name': '...",True,False,"['amtsgericht', 'memmingen', 'abteilung', 'zwa...",amtsgericht memmingen\nabteilung für zwangsvol...,163464253141,Julia Zauner
1,Amtsgericht Frankfurt am Main\nMobiliarzwangsv...,01.06.2022/1654077659_Doc_01062022_000000001_1...,approved_seizure,amtsgericht frankfurt am main\nmobiliarzwangsv...,"{'case_slug': '110872558124', 'debtor_name': '...",True,False,"['amtsgericht', 'frankfurt', 'main', 'mobiliar...",amtsgericht frankfurt am main\nmobiliarzwangsv...,110872558124,"Kaiser, Vanessa"
2,Amtsgericht Eschweiler\n070071 967072\n-Geschä...,01.06.2022/1654077662_Doc_01062022_000000001_1...,approved_seizure,amtsgericht eschweiler\n070071 967072\n-geschä...,"{'case_slug': '114249164962', 'debtor_name': '...",True,False,"['amtsgericht', 'eschweiler', '-geschäftsstell...",amtsgericht eschweiler\n<NUMBER><PHONE>\n-gesc...,114249164962,Pitschel
3,Amtsgericht Münster\n-33- Amtsgericht Münster ...,01.07.2024/ocr-v2_BB_005_01072024_122825.pdf,approved_seizure,amtsgericht münster\n-33- amtsgericht münster ...,"{'case_slug': '136916228754', 'debtor_name': '...",True,False,"['amtsgericht', 'münster', 'amtsgericht', 'mün...",amtsgericht münster\n<NUMBER>- amtsgericht mün...,136916228754,Georgina Kaiser
4,Amtsgericht Viersen\n-Geschäftsstelle-\n-15- A...,01.07.2025/ocr-v2_Allgemeines_Schreiben_R__202...,approved_attachment_and_transfer_order,amtsgericht viersen\n-geschäftsstelle-\n-15- a...,"{'case_slug': '125495503778', 'debtor_name': '...",True,False,"['amtsgericht', 'viersen', '-geschäftsstelle-'...",amtsgericht viersen\n-geschäftsstelle-\n<NUMBE...,125495503778,Sultan Aslan
...,...,...,...,...,...,...,...,...,...,...,...
439,Amtsgericht Münster\n-33- Amtsgericht Münster ...,30.10.2025/ocr-v2_c0872fd7aa1c431e.pdf,approved_attachment_and_transfer_order,amtsgericht münster\n-33- amtsgericht münster ...,"{'case_slug': '181060314323', 'debtor_name': '...",True,False,"['amtsgericht', 'münster', 'amtsgericht', 'mün...",amtsgericht münster\n<NUMBER>- amtsgericht mün...,181060314323,Anita Große Leusbrock
440,Amtsgericht Wuppertal\n-44- Amtsgericht Wupper...,31.01.2023/1675166710_YA_003_31012023_124814.pdf,approved_seizure,amtsgericht wuppertal\n-44- amtsgericht wupper...,"{'case_slug': '110454161073', 'debtor_name': '...",True,False,"['amtsgericht', 'wuppertal', 'amtsgericht', 'w...",amtsgericht wuppertal\n<NUMBER>- amtsgericht w...,110454161073,Magner
441,Amtsgericht Leverkusen\n-45- Amtsgericht Lever...,31.05.2023/1685534803_PF_005_31052023_140508.pdf,approved_seizure,amtsgericht leverkusen\n-45- amtsgericht lever...,"{'case_slug': '117578968598', 'debtor_name': '...",True,False,"['amtsgericht', 'leverkusen', 'amtsgerichen', ...",amtsgericht leverkusen\n<NUMBER>- amtsgericht ...,117578968598,Aschenbach
442,Amtsgericht Frankfurt am Main\nMobiliarzwangsv...,31.07.2023/1690794316_KD_004_31072023_110320.pdf,approved_seizure,amtsgericht frankfurt am main\nmobiliarzwangsv...,"{'case_slug': '125677855012', 'debtor_name': '...",True,False,"['amtsgericht', 'frankfurt', 'main', 'mobiliar...",amtsgericht frankfurt am main\nmobiliarzwangsv...,125677855012,"Tiropoulos, Konstantinos"


In [14]:
not_start_with_one = data[data['case_slug'].apply(lambda x: not x.startswith('1') if isinstance(x, str) else True)]
not_start_with_one

,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,tokenized,text_w_tags,case_slug,debtor_name
373,Amtsgericht Heinsberg\n-10- Amtsgericht Heinsb...,27.06.2023/1687860442_KD_007_27062023_115318.pdf,approved_seizure,amtsgericht heinsberg\n-10- amtsgericht heinsb...,"{'case_slug': '02452109154', 'debtor_name': 'B...",True,False,"['amtsgericht', 'heinsberg', 'amtsgericht', 'h...",amtsgericht heinsberg\n<NUMBER>- amtsgericht h...,02452109154,Bauer
430,Amtsgericht Langen (Hessen)\nVollstreckungsger...,30.05.2023/1685448395_DS_007_30052023_134146.pdf,approved_seizure,amtsgericht langen (hessen)\nvollstreckungsger...,"{'case_slug': '2000', 'debtor_name': 'Rothen, ...",True,False,"['amtsgericht', 'lang', 'hessen', 'vollstrecku...",amtsgericht langen (hessen)\nvollstreckungsger...,2000,"Rothen, Vanessa"


In [15]:
print_out_clickable_link(not_start_with_one)

Index: 373, Case Slug: 02452109154, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=27.06.2023/1687860442_KD_007_27062023_115318.pdf
Index: 430, Case Slug: 2000, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=30.05.2023/1685448395_DS_007_30052023_134146.pdf


In [16]:
# drop
data = data.drop(index=373).reset_index(drop=True)

### 1) Extracting SLUG Parameter from Document

In [17]:
print("All of them starts with 1: ", all(data['case_slug'].apply(lambda x: x.startswith('1'))))
print("All of them have length 12: ", all(data['case_slug'].apply(lambda x: len(x)==12 or len(x)==11 or len(x)==4)))

All of them starts with 1:  False
All of them have length 12:  True


In [18]:
# USE CURRENT SLUG EXTRACTOR FOR EXTRACTION
import os

notebook_dir = os.getcwd()
project_dir = os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(notebook_dir))))
print("Notebook Directory:", notebook_dir)
print("Project Directory:", project_dir)
os.chdir(project_dir)


Notebook Directory: /Users/melih.gorgulu/Desktop/Projects/intent_recognition/notebooks/after-court/pfub-erlass/param_extraction
Project Directory: /Users/melih.gorgulu/Desktop/Projects/intent_recognition


In [19]:
from src.services.attachment_processing.aftercourt_extractors.ladung.slug_extractor import SlugExtractor
slug_extractor = SlugExtractor()
data['extracted_slug'] = data['cleaned_text'].apply(lambda x: slug_extractor.extract(x))

In [20]:
# first check none values
slug_cannot_extracted = data[data['extracted_slug'].isnull()]

In [21]:
print_out_clickable_link(slug_cannot_extracted)
# 55: Case Slug: 107431129462, in document its: 10d7431129462, document date= 4 JAN 2022
# 84: Case Slug: 117874546452, in document its: 11874546452, document date= 25 09 2023 -> check why we cannot extract
# 136-> document do not have slug info, so the method is correct
# 153-> slug looks weird: lhr Zeichen: 1 1990/20SK01/SK105745046596, DATE  02.01.2024-> ask to court staff if this type of slugs are valid, and may still happen
# 200-> lhrZeichen: 11554/21SK01 / SK 10653146984
# 217 -> lhrZeichen: 11554/21SK01 / SK 10653146984
# 262 -> document have slug: 11680109166. Check why we cannot extract
# 318 -> document have this slug: 22221/225K01


Index: 55, Case Slug: 107431129462, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=05.01.2022/1641394840_Doc_05012022_000000001_154522_1.pdf
Index: 84, Case Slug: 117874546452, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=06.10.2023/1696584817_KD_003_06102023_111057.pdf
Index: 136, Case Slug: 115330475621, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=10.10.2023/1696925100_KD_001_10102023_095401.pdf
Index: 153, Case Slug: 105745046596, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=12.01.2024/ocr-v2_JD_002_12012024_085742.pdf
Index: 200, Case Slug: 10653146984, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=16.08.2022/Scan_YA_00416082022_123713.pdf
Index: 217, Case Slug: 10653146984, Link: https://eu-central-1.console.aws.

In [22]:
indexes_to_check = [84,262]
cannot_extract = data[data.index.isin(indexes_to_check)]
cannot_extract

,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,tokenized,text_w_tags,case_slug,debtor_name,extracted_slug
84,Amtsgericht Velbert\nPAIR Finance GmbH\n-Gesch...,06.10.2023/1696584817_KD_003_06102023_111057.pdf,approved_seizure,amtsgericht velbert\npair finance gmbh\n-gesch...,"{'case_slug': '117874546452', 'debtor_name': '...",True,False,"['amtsgericht', 'velbert', 'pair', 'finance', ...",amtsgericht velbert\npair finance gmbh\n-gesch...,117874546452,"Arik, Emre",None
262,"Amtsgericht Kiel\nAmtsgericht Kiel, Deliusstra...",20.03.2023/1679313984_AA_001_20032023_125315.pdf,approved_seizure,"amtsgericht kiel\namtsgericht kiel, deliusstra...","{'case_slug': '11680109166', 'debtor_name': No...",True,False,"['amtsgericht', 'kiel', 'amtsgericht', 'kiel',...","amtsgericht kiel\namtsgericht kiel, deliusstra...",11680109166,None,None


In [23]:
def print_slug_in_text_with_color(text, slug):
    # ANSI color codes
    BOLD = '\033[1m'
    CYAN = '\033[96m'
    YELLOW = '\033[93m'
    RED = '\033[91m'
    GREEN = '\033[92m'
    RESET = '\033[0m'
    
    # Escape special characters in slug for regex
    print(f"{BOLD}{CYAN}{'='*80}{RESET}")
    print(f"{BOLD}{YELLOW}SEARCHING FOR SLUG:{RESET} {GREEN}{slug}{RESET}")
    print(f"{BOLD}{CYAN}{'='*80}{RESET}\n")
    
    escaped_slug = re.escape(slug)
    # Use regex to find and wrap the slug with ANSI color codes
    highlighted_text = re.sub(f'({escaped_slug})', f'{BOLD}{RED}\\1{RESET}', text)
    print(highlighted_text)
    print(f"\n{BOLD}{CYAN}{'='*80}{RESET}")

print_slug_in_text_with_color(cannot_extract.iloc[0]['cleaned_text'], cannot_extract.iloc[0]['case_slug'])

SEARCHING FOR SLUG: 117874546452

amtsgericht velbert
pair finance gmbh
-geschäftsstelle-
05. okt. 2023
-15- amtsgericht velbert - postfach 101380 42513 velbert
25.09.2023
seite 1 von 1
pair finance gmbh
vertr. d. d. gf
aktenzeichen
hardenbergstraße 32
15 m 1424/23
bei antwort bitte angeben
10623 berlin
bearbeiter
frau dahmen
durchwahl
02051945-165
ihr zeichen: 11874546452
sehr geehrte damen und herren,
in der zwangsvollstreckungssache
strato ag gegen arik
wird ihnen mitgeteilt, dass dem antrag entsprochen wurde und die
zustellung des pfändungs- und überweisungsbeschlusses an
schuldner und drittschuldner veranlasst worden ist.
anschrift
nedderstraße 40
anliegend erhalten sie ihre unterlagen zurück.
42549 velbert
sprechzeiten
mo fr 8:00 12:00 uhr; do
13:30 14:30 uhr
mit freundlichen grüßen
telefon
020519450
auf anordnung
telefax:
schmitz-jansen
02051945199
justizbeschäftigte
www.ag-velbert.nrw.de
automatisiert erstellt, ohne unterschrift gültig
nachtbriefkasten: nedderstraße
40, 42549 v

In [24]:
print_out_clickable_link(cannot_extract)

Index: 84, Case Slug: 117874546452, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=06.10.2023/1696584817_KD_003_06102023_111057.pdf
Index: 262, Case Slug: 11680109166, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=20.03.2023/1679313984_AA_001_20032023_125315.pdf


In [25]:
# lets run extraction on this to see why its not working
slug_extractor.extract(cannot_extract.iloc[0]['cleaned_text'])

#### The reason why we cannot extract the slug in this cases: they are 11 digit, our pattern search for 12-13 digit

In [26]:
slug_extractor.SLUG_PATTERN = re.compile(r'\b1[0-9]{10,12}\b')

In [27]:
print(f"After including 11 digit: {slug_extractor.extract(cannot_extract.iloc[0]['cleaned_text'])}")
print(f"After including 11 digit: {slug_extractor.extract(cannot_extract.iloc[1]['cleaned_text'])}")

After including 11 digit: 11874546452
After including 11 digit: 11680109166


In [28]:
data['extracted_slug_new'] = data['cleaned_text'].apply(lambda x: slug_extractor.extract(x))

In [29]:
old_and_new_differ = data[data['extracted_slug_new']!=data['extracted_slug']]
old_and_new_differ


,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,tokenized,text_w_tags,case_slug,debtor_name,extracted_slug,extracted_slug_new
55,4 070071 967072\nAmtsgericht Kiel\nAmtsgericht...,05.01.2022/1641394840_Doc_05012022_000000001_1...,approved_seizure,4 070071 967072\namtsgericht kiel\namtsgericht...,"{'case_slug': '107431129462', 'debtor_name': N...",True,False,"['amtsgericht', 'kiel', 'amtsgericht', 'kiel',...",<NUMBER> <NUMBER><PHONE>\namtsgericht kiel\nam...,107431129462,None,None,None
84,Amtsgericht Velbert\nPAIR Finance GmbH\n-Gesch...,06.10.2023/1696584817_KD_003_06102023_111057.pdf,approved_seizure,amtsgericht velbert\npair finance gmbh\n-gesch...,"{'case_slug': '117874546452', 'debtor_name': '...",True,False,"['amtsgericht', 'velbert', 'pair', 'finance', ...",amtsgericht velbert\npair finance gmbh\n-gesch...,117874546452,"Arik, Emre",None,11874546452
136,"Amtsgericht Rendsburg\nAmtsgericht Rendsburg, ...",10.10.2023/1696925100_KD_001_10102023_095401.pdf,approved_seizure,"amtsgericht rendsburg\namtsgericht rendsburg, ...","{'case_slug': '115330475621', 'debtor_name': '...",True,False,"['amtsgericht', 'rendsburg', 'amtsgericht', 'r...","amtsgericht rendsburg\namtsgericht rendsburg, ...",115330475621,Jan Papendorf,None,None
153,Amtsgericht Viersen\n-15- Amtsgericht Viersen ...,12.01.2024/ocr-v2_JD_002_12012024_085742.pdf,approved_attachment_and_transfer_order,amtsgericht viersen\n-15- amtsgericht viersen ...,"{'case_slug': '105745046596', 'debtor_name': '...",True,False,"['amtsgericht', 'viersen', 'amtsgericht', 'vie...",amtsgericht viersen\n<NUMBER>- amtsgericht vie...,105745046596,Kevin Käpplinger,None,None
200,Amtsgericht Heinsberg\n4070071967072\n-10- Amt...,16.08.2022/Scan_YA_00416082022_123713.pdf,approved_seizure,amtsgericht heinsberg\n4070071967072\n-10- amt...,"{'case_slug': '10653146984', 'debtor_name': 'M...",True,False,"['amtsgericht', 'heinsberg', 'amtsgericht', 'h...",amtsgericht heinsberg\n<NUMBER>\n<NUMBER>- amt...,10653146984,Müller,None,10653146984
217,Amtsgericht Heinsberg\n4070071967072\n-10- Amt...,17.08.2022/Scan_YA_00416082022_123713_1.pdf,approved_seizure,amtsgericht heinsberg\n4070071967072\n-10- amt...,"{'case_slug': '10653146984', 'debtor_name': 'M...",True,False,"['amtsgericht', 'heinsberg', 'amtsgericht', 'h...",amtsgericht heinsberg\n<NUMBER>\n<NUMBER>- amt...,10653146984,Müller,None,10653146984
262,"Amtsgericht Kiel\nAmtsgericht Kiel, Deliusstra...",20.03.2023/1679313984_AA_001_20032023_125315.pdf,approved_seizure,"amtsgericht kiel\namtsgericht kiel, deliusstra...","{'case_slug': '11680109166', 'debtor_name': No...",True,False,"['amtsgericht', 'kiel', 'amtsgericht', 'kiel',...","amtsgericht kiel\namtsgericht kiel, deliusstra...",11680109166,None,None,11680109166
318,"Amtsgericht Kiel\nAmtsgericht Kiel, Deliusstra...",24.07.2023/1690187636_KD_004_24072023_101425.pdf,approved_attachment_and_transfer_order,"amtsgericht kiel\namtsgericht kiel, deliusstra...","{'case_slug': '109682166656', 'debtor_name': '...",True,False,"['amtsgericht', 'kiel', 'amtsgericht', 'kiel',...","amtsgericht kiel\namtsgericht kiel, deliusstra...",109682166656,"Abdine, Ayman",None,None


### COMPARE: OCR CASE SLUG VS OUR EXTRACTION

In [30]:
different = data[data['case_slug']!=data['extracted_slug_new']]

In [31]:
# this indexes are okay to be different: 55,136,153,200,217,31
# indexes to check = 84, 317, 318, 429
different

,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,tokenized,text_w_tags,case_slug,debtor_name,extracted_slug,extracted_slug_new
55,4 070071 967072\nAmtsgericht Kiel\nAmtsgericht...,05.01.2022/1641394840_Doc_05012022_000000001_1...,approved_seizure,4 070071 967072\namtsgericht kiel\namtsgericht...,"{'case_slug': '107431129462', 'debtor_name': N...",True,False,"['amtsgericht', 'kiel', 'amtsgericht', 'kiel',...",<NUMBER> <NUMBER><PHONE>\namtsgericht kiel\nam...,107431129462,None,None,None
84,Amtsgericht Velbert\nPAIR Finance GmbH\n-Gesch...,06.10.2023/1696584817_KD_003_06102023_111057.pdf,approved_seizure,amtsgericht velbert\npair finance gmbh\n-gesch...,"{'case_slug': '117874546452', 'debtor_name': '...",True,False,"['amtsgericht', 'velbert', 'pair', 'finance', ...",amtsgericht velbert\npair finance gmbh\n-gesch...,117874546452,"Arik, Emre",None,11874546452
136,"Amtsgericht Rendsburg\nAmtsgericht Rendsburg, ...",10.10.2023/1696925100_KD_001_10102023_095401.pdf,approved_seizure,"amtsgericht rendsburg\namtsgericht rendsburg, ...","{'case_slug': '115330475621', 'debtor_name': '...",True,False,"['amtsgericht', 'rendsburg', 'amtsgericht', 'r...","amtsgericht rendsburg\namtsgericht rendsburg, ...",115330475621,Jan Papendorf,None,None
153,Amtsgericht Viersen\n-15- Amtsgericht Viersen ...,12.01.2024/ocr-v2_JD_002_12012024_085742.pdf,approved_attachment_and_transfer_order,amtsgericht viersen\n-15- amtsgericht viersen ...,"{'case_slug': '105745046596', 'debtor_name': '...",True,False,"['amtsgericht', 'viersen', 'amtsgericht', 'vie...",amtsgericht viersen\n<NUMBER>- amtsgericht vie...,105745046596,Kevin Käpplinger,None,None
317,Amtsgericht Mönchengladbach\n-22- Amtsgericht ...,24.07.2023/1690187635_KD_004_24072023_101422.pdf,approved_attachment_and_transfer_order,amtsgericht mönchengladbach\n-22- amtsgericht ...,"{'case_slug': '109250362497', 'debtor_name': '...",True,False,"['amtsgericht', 'mönchengladbach', 'amtsgerich...",amtsgericht mönchengladbach\n<NUMBER>- amtsger...,109250362497,Selhattin Yazmalar,1092250362497,1092250362497
318,"Amtsgericht Kiel\nAmtsgericht Kiel, Deliusstra...",24.07.2023/1690187636_KD_004_24072023_101425.pdf,approved_attachment_and_transfer_order,"amtsgericht kiel\namtsgericht kiel, deliusstra...","{'case_slug': '109682166656', 'debtor_name': '...",True,False,"['amtsgericht', 'kiel', 'amtsgericht', 'kiel',...","amtsgericht kiel\namtsgericht kiel, deliusstra...",109682166656,"Abdine, Ayman",None,None
429,Amtsgericht Langen (Hessen)\nVollstreckungsger...,30.05.2023/1685448395_DS_007_30052023_134146.pdf,approved_seizure,amtsgericht langen (hessen)\nvollstreckungsger...,"{'case_slug': '2000', 'debtor_name': 'Rothen, ...",True,False,"['amtsgericht', 'lang', 'hessen', 'vollstrecku...",amtsgericht langen (hessen)\nvollstreckungsger...,2000,"Rothen, Vanessa",114819775940,114819775940


In [32]:
check = different[different.index.isin([84,317,318,429])]
print_out_clickable_link(check)

Index: 84, Case Slug: 117874546452, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=06.10.2023/1696584817_KD_003_06102023_111057.pdf
Index: 317, Case Slug: 109250362497, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=24.07.2023/1690187635_KD_004_24072023_101422.pdf
Index: 318, Case Slug: 109682166656, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=24.07.2023/1690187636_KD_004_24072023_101425.pdf
Index: 429, Case Slug: 2000, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=30.05.2023/1685448395_DS_007_30052023_134146.pdf


In [33]:
check

,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,tokenized,text_w_tags,case_slug,debtor_name,extracted_slug,extracted_slug_new
84,Amtsgericht Velbert\nPAIR Finance GmbH\n-Gesch...,06.10.2023/1696584817_KD_003_06102023_111057.pdf,approved_seizure,amtsgericht velbert\npair finance gmbh\n-gesch...,"{'case_slug': '117874546452', 'debtor_name': '...",True,False,"['amtsgericht', 'velbert', 'pair', 'finance', ...",amtsgericht velbert\npair finance gmbh\n-gesch...,117874546452,"Arik, Emre",None,11874546452
317,Amtsgericht Mönchengladbach\n-22- Amtsgericht ...,24.07.2023/1690187635_KD_004_24072023_101422.pdf,approved_attachment_and_transfer_order,amtsgericht mönchengladbach\n-22- amtsgericht ...,"{'case_slug': '109250362497', 'debtor_name': '...",True,False,"['amtsgericht', 'mönchengladbach', 'amtsgerich...",amtsgericht mönchengladbach\n<NUMBER>- amtsger...,109250362497,Selhattin Yazmalar,1092250362497,1092250362497
318,"Amtsgericht Kiel\nAmtsgericht Kiel, Deliusstra...",24.07.2023/1690187636_KD_004_24072023_101425.pdf,approved_attachment_and_transfer_order,"amtsgericht kiel\namtsgericht kiel, deliusstra...","{'case_slug': '109682166656', 'debtor_name': '...",True,False,"['amtsgericht', 'kiel', 'amtsgericht', 'kiel',...","amtsgericht kiel\namtsgericht kiel, deliusstra...",109682166656,"Abdine, Ayman",None,None
429,Amtsgericht Langen (Hessen)\nVollstreckungsger...,30.05.2023/1685448395_DS_007_30052023_134146.pdf,approved_seizure,amtsgericht langen (hessen)\nvollstreckungsger...,"{'case_slug': '2000', 'debtor_name': 'Rothen, ...",True,False,"['amtsgericht', 'lang', 'hessen', 'vollstrecku...",amtsgericht langen (hessen)\nvollstreckungsger...,2000,"Rothen, Vanessa",114819775940,114819775940


# RESULT: AMONG 4 DIFFERENT CASES, OUR EXTRACTION IS CORRECT